# ETL para Power BI

**Desarrollado por:** Alan Arturo Hernández Gutiérrez

En este notebook preparo los datos y genero los dos archivos de Power BI.
Ejecutar las celdas en orden.

## 01. Librerías

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

## 02. Carpetas del proyecto

La carpeta raíz se obtiene del directorio de ejecución; ejecutar desde la raíz del proyecto o desde `notebooks/`.

In [ ]:
BASE = Path.cwd().resolve()
if BASE.name.lower() == "notebooks":
    BASE = BASE.parent

INPUT = BASE / "input"
CATALOGOS = BASE / "catalogos"
OUTPUT = BASE / "output"

OUTPUT.mkdir(parents=True, exist_ok=True)

archivo_clientes = INPUT / "Bank Customer Churn Prediction.csv"
archivo_paises = CATALOGOS / "DimPais.csv"
archivo_escenarios = CATALOGOS / "SupuestosEscenarios.json"

## 03. Lectura de insumos

In [ ]:
clientes = pd.read_csv(archivo_clientes, encoding="utf-8-sig")
paises = pd.read_csv(archivo_paises, encoding="utf-8-sig")

with open(archivo_escenarios, encoding="utf-8") as archivo:
    escenarios = json.load(archivo)

print("Clientes:", len(clientes))
print("Bajas:", int(clientes["churn"].sum()))

## 04. Revisar la base

Comprobar que trabajamos con los registros y columnas esperados.

In [ ]:
columnas_originales = [
    "customer_id", "credit_score", "country", "gender",
    "age", "tenure", "balance", "products_number",
    "credit_card", "active_member", "estimated_salary", "churn"
]

assert clientes.columns.tolist() == columnas_originales
assert len(clientes) == 10000
assert clientes["customer_id"].is_unique
assert int(clientes["churn"].sum()) == 2037

print("Base revisada")

## 05. Copiar las columnas originales

In [ ]:
fact = clientes[columnas_originales].copy()

fact["saldo_estado"] = np.where(
    fact["balance"] == 0, "Cero", "Positivo"
)

## 06. Agrupar las edades

In [ ]:
limites_edad = [17, 29, 39, 49, 59, 69, np.inf]
etiquetas_edad = ["18–29", "30–39", "40–49", "50–59", "60–69", "70+"]

fact["edad_banda"] = pd.cut(
    fact["age"], bins=limites_edad,
    labels=etiquetas_edad, include_lowest=True
).astype(str)

## 07. Calcular los cuartiles del saldo

El saldo cero se conserva como un grupo independiente.

In [ ]:
saldos_positivos = fact.loc[fact["balance"] > 0, "balance"]
cuartiles = saldos_positivos.quantile([0.25, 0.50, 0.75])

q1 = float(cuartiles.loc[0.25])
q2 = float(cuartiles.loc[0.50])
q3 = float(cuartiles.loc[0.75])

limites_saldo = [0, q1, q2, q3, np.inf]
etiquetas_saldo = [
    f"Positivo Q1 (≤{q1:.2f})",
    f"Positivo Q2 (≤{q2:.2f})",
    f"Positivo Q3 (≤{q3:.2f})",
    f"Positivo Q4 (>{q3:.2f})"
]

## 08. Crear las bandas de saldo

In [ ]:
fact["saldo_banda"] = "Cero"
tiene_saldo = fact["balance"] > 0

fact.loc[tiene_saldo, "saldo_banda"] = pd.cut(
    fact.loc[tiene_saldo, "balance"],
    bins=limites_saldo,
    labels=etiquetas_saldo,
    include_lowest=True
).astype(str)

print("Cuartiles:", q1, q2, q3)

## 09. Tipo de vinculación y actividad

In [ ]:
fact["vinculacion_tipo"] = np.where(
    fact["products_number"] == 1, "Un producto", "Multiproducto"
)

actividad = fact["active_member"].map({0: "Inactivo", 1: "Activo"})

fact["perfil_actividad_saldo"] = actividad + " / " + fact["saldo_estado"]
fact["perfil_actividad_productos"] = actividad + " / " + fact["vinculacion_tipo"]
fact["perfil_saldo_producto"] = fact["saldo_estado"] + " / " + fact["vinculacion_tipo"]
fact["productos_alto_3o4"] = (fact["products_number"] >= 3).astype("int8")

## 10. Asignar H1, H2 y H3

In [ ]:
h1 = (fact["active_member"] == 0) & (fact["products_number"] == 1)
h2 = fact["products_number"] >= 3
h3 = (fact["active_member"] == 1) & (fact["products_number"] == 1)

fact["hipotesis"] = np.select(
    [h1, h2, h3], ["H1", "H2", "H3"], default="Fuera"
)

fact["simulable"] = fact["hipotesis"].isin(["H1", "H3"]).astype("int8")

print(fact["hipotesis"].value_counts())

## 11. Preparar controles de calidad

In [ ]:
calidad = pd.DataFrame({
    "customer_id": fact["customer_id"],
    "tenure_cero": (fact["tenure"] == 0).astype("int8"),
    "score_extremo_350_850": fact["credit_score"].isin([350, 850]).astype("int8"),
    "salario_menor_100": (fact["estimated_salary"] < 100).astype("int8"),
    "edad_menos_antiguedad_menor_18": (
        (fact["age"] - fact["tenure"]) < 18
    ).astype("int8")
})

## 12. Resumir los grupos comerciales

In [ ]:
hipotesis = fact.groupby("hipotesis", dropna=False).agg(
    clientes=("customer_id", "size"),
    bajas=("churn", "sum"),
    saldo_observado=("balance", "sum")
).reset_index()

hipotesis["churn_pct"] = 100 * hipotesis["bajas"] / hipotesis["clientes"]
hipotesis["contribucion_bajas_pct"] = 100 * hipotesis["bajas"] / 2037

hipotesis

## 13. Definir las variables del Radar

Se usan variables individuales y algunos cruces entre ellas.

In [ ]:
variables_radar = [
    "country", "active_member", "products_number", "credit_card",
    "edad_banda", "saldo_estado", "vinculacion_tipo",
    "perfil_actividad_saldo", "perfil_actividad_productos",
    "perfil_saldo_producto"
]

cruces_radar = [
    ("country", "active_member"),
    ("country", "products_number"),
    ("active_member", "products_number"),
    ("edad_banda", "active_member"),
    ("edad_banda", "products_number"),
    ("saldo_estado", "active_member"),
    ("saldo_estado", "products_number"),
    ("credit_card", "active_member")
]

reglas_radar = [(variable,) for variable in variables_radar] + cruces_radar

## 14. Resumir un segmento

Esta función calcula las cifras que se mostrarán en el Radar.

In [ ]:
def resumen_radar(columnas, valores, grupo, codigo):
    cantidad = len(grupo)
    bajas = int(grupo["churn"].sum())
    tasa = bajas / cantidad

    cuadrante = "Tasa alta" if tasa > 0.2037 else "Tasa baja"
    cuadrante += " / Volumen alto" if bajas >= 204 else " / Volumen bajo"

    return {
        "segmento_id": codigo,
        "regla": " | ".join(f"{a}={b}" for a, b in zip(columnas, valores)),
        "estructura": "+".join(columnas),
        "clientes": cantidad,
        "bajas": bajas,
        "churn_pct": round(100 * tasa, 4),
        "contribucion_bajas_pct": round(100 * bajas / 2037, 4),
        "participacion_clientes_pct": round(100 * cantidad / 10000, 4),
        "indice_relativo_churn": round(tasa / 0.2037, 4),
        "exceso_descriptivo": round(bajas - cantidad * 0.2037, 3),
        "saldo_asociado_bajas": round(grupo.loc[grupo.churn.eq(1), "balance"].sum(), 2),
        "cuadrante": cuadrante
    }

## 15. Crear los segmentos

Se conservan los grupos con 100 clientes o más.

In [ ]:
segmentos = []
pertenencias = []
numero_segmento = 0

for columnas in reglas_radar:
    for valores, grupo in fact.groupby(list(columnas), dropna=False):
        if len(grupo) < 100:
            continue

        if not isinstance(valores, tuple):
            valores = (valores,)

        numero_segmento += 1
        codigo = f"R{numero_segmento:03}"

        segmentos.append(resumen_radar(columnas, valores, grupo, codigo))
        pertenencias.extend((codigo, int(cliente)) for cliente in grupo["customer_id"])

## 16. Organizar las tablas del Radar

In [ ]:
radar = pd.DataFrame(segmentos)
radar = radar.sort_values(["bajas", "clientes"], ascending=False)

radar_clientes = pd.DataFrame(
    pertenencias, columns=["segmento_id", "customer_id"]
)

print("Segmentos:", len(radar))
print("Pertenencias:", len(radar_clientes))

## 17. Calcular los contactos

Primero se calcula cuántos contactos permite la cobertura y el presupuesto.

In [ ]:
def calcular_contactos(meta, disponible, costo_contacto, costo_fijo):
    if meta == 0 or disponible < costo_fijo + costo_contacto:
        return 0

    if costo_contacto == 0 or math.isinf(disponible):
        return meta

    posibles = math.floor((disponible - costo_fijo) / costo_contacto + 1e-10)
    return min(meta, posibles)

## 18. Calcular un escenario

El presupuesto disponible se aplica primero a H1 y después a H3.

In [ ]:
def calcular_escenario(cobertura, mejora, costo_contacto, costo_fijo,
                      margen, cac, reemplazo, presupuesto=None):
    disponible = math.inf if presupuesto is None else presupuesto
    resultados = []

    for grupo in ["H1", "H3"]:
        datos = hipotesis.loc[hipotesis["hipotesis"] == grupo].iloc[0]
        poblacion = int(datos["clientes"])
        meta = math.floor(poblacion * cobertura + 1e-10)
        contactos = calcular_contactos(meta, disponible, costo_contacto, costo_fijo)

        costo = costo_fijo + contactos * costo_contacto if contactos else 0
        retenidos = contactos * mejora
        disponible -= costo

        resultados.append({
            "hipotesis": grupo,
            "poblacion": poblacion,
            "contactos": contactos,
            "retenciones_incrementales_esperadas": round(retenidos, 3),
            "costo": costo,
            "beneficio_margen": round(retenidos * margen, 2),
            "beneficio_cac_alternativo": round(retenidos * reemplazo * cac, 2)
        })

    return resultados

## 19. Calcular los escenarios del catálogo

In [ ]:
filas_escenarios = []

for caso in escenarios:
    resultados = calcular_escenario(
        caso["cobertura"], caso["mejora"],
        caso["costo_contacto"], caso["costo_fijo"],
        caso["margen"], caso["cac"], caso["reemplazo"]
    )

    for resultado in resultados:
        filas_escenarios.append({"escenario": caso["escenario"], **resultado})

escenarios_tabla = pd.DataFrame(filas_escenarios)
escenarios_neto = escenarios_tabla["beneficio_margen"] - escenarios_tabla["costo"]
escenarios_cac = escenarios_tabla["beneficio_cac_alternativo"] - escenarios_tabla["costo"]

escenarios_tabla["neto_margen"] = escenarios_neto
escenarios_tabla["neto_cac_alternativo"] = escenarios_cac

## 20. Revisión final de cifras

Antes de guardar, verificar que coinciden con los resultados del análisis.

In [ ]:
assert fact.shape == (10000, 22)
assert int(fact["churn"].sum()) == 2037
assert len(radar) == 84
assert len(radar_clientes) == 178915
assert radar_clientes["customer_id"].nunique() == 10000
assert not radar_clientes.duplicated().any()

print("Cifras correctas")

## 21. Guardar el Excel para Power BI

Se crean las seis hojas del modelo.

In [ ]:
ruta_excel = OUTPUT / "Modelo_Analitica_Clientes.xlsx"

hojas = {
    "FactClientes": fact,
    "DimPais": paises,
    "RadarSegmentos": radar,
    "CalidadCliente": calidad,
    "Escenarios": escenarios_tabla,
    "HipotesisResumen": hipotesis
}

with pd.ExcelWriter(ruta_excel, engine="openpyxl") as escritor:
    for nombre, tabla in hojas.items():
        tabla.to_excel(escritor, sheet_name=nombre, index=False)

## 22. Guardar el CSV del Radar

In [ ]:
ruta_radar = OUTPUT / "RadarCliente.csv"
radar_clientes.to_csv(ruta_radar, index=False, encoding="utf-8-sig")

print("Archivos finales:")
print("-", ruta_excel.name)
print("-", ruta_radar.name)

## 23. Comprobar los archivos generados

In [ ]:
assert ruta_excel.is_file()
assert ruta_radar.is_file()

print("ETL terminado. Los archivos están en:")
print(OUTPUT)